In [1]:
%pip install transformers datasets accelerate bitsandbytes peft trl torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, set_seed

model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Add special tokens for function calling
special_tokens = ["<think>", "</think>", "<function>", "</function>"]

# Add to tokenizer
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

print(f"✅ Added special tokens: {special_tokens}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Test tokenization of special tokens
test_text = "<think>This is a test</think>"
tokens = tokenizer.tokenize(test_text)
print(f"Tokenized '<think>This is a test</think>': {tokens}")

✅ Added special tokens: ['<think>', '</think>', '<function>', '</function>']
Tokenizer vocab size: 128260
Tokenized '<think>This is a test</think>': ['<think>', 'This', 'Ġis', 'Ġa', 'Ġtest', '</think>']


In [4]:
def prepare_dataset(csv_path):
    df = pd.read_csv(csv_path)
    
    conversations = []
    for _, row in df.iterrows():
        # Include patient ID in the user query
        user_query_with_id = f"Patient ID: {row['patient_id']}\nQuery: {row['patient_query']}"
        
        conversation = {
            "messages": [
                {
                    "role": "system",
                    "content": "You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function."
                },
                {
                    "role": "user",
                    "content": user_query_with_id
                },
                {
                    "role": "assistant", 
                    "content": row['assistant_response']
                }
            ]
        }
        conversations.append(conversation)
    
    return Dataset.from_list(conversations)

def preprocess(sample):
    messages = sample["messages"]
    # Use LLaMA's built-in chat template - it's already optimized
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

In [5]:
# # Load and prepare your dataset
dataset_path = "/home/jovyan/Works/MSc_Thesis/dataset/mh_function_calling_dataset.csv"
dataset = prepare_dataset(dataset_path)
dataset = dataset.map(preprocess, remove_columns=dataset.column_names)

Map: 100%|██████████| 4000/4000 [00:00<00:00, 14933.86 examples/s]


In [6]:
# Split the dataset into train and test (90% train, 10% test)
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split["train"]
test_dataset = dataset_split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 3200
Test samples: 800


In [7]:
# Let's look at how we formatted the dataset
print(train_dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Jul 2025

You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.<|eot_id|><|start_header_id|>user<|end_header_id|>

Patient ID: UID_144
Query: My anxiety medication may be affecting my BP for 10 days<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<think>
The patient's blood pressure concerns in relation to their mental health need monitoring over the specified timeframe. I should extract their patient ID and number of days to get the appropriate blood pressure readings.
</think>

<function>
{
  "name": "get_blood_pressure_data",
  "parameters": {
    "patient_id": "UID_144",
    "num_days": 10


In [8]:
# Check tokenization
sample_tokens = tokenizer.tokenize(dataset[1]["text"])
print(f"Tokenized length: {len(sample_tokens)} tokens")

Tokenized length: 214 tokens


In [9]:
# You can use full precision instead of 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # Full precision
    device_map="auto",
    # No need for load_in_4bit=True
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.88it/s]


In [10]:
# IMPORTANT: Resize embeddings for new special tokens
model.resize_token_embeddings(len(tokenizer))


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(128260, 3072)

#### Configure the LoRA

In [11]:
from peft import LoraConfig, TaskType

rank_dimension = 16  # Good balance for 3B model
lora_alpha = 64      # Lower than their 64 - more conservative for function calling
lora_dropout = 0.05   # Slightly higher to prevent overfitting on your smaller dataset

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers (most important)
        "gate_proj", "up_proj", "down_proj",     # MLP layers
        "embed_tokens", "lm_head"              # Uncomment if you need stronger adaptation
    ],
    task_type=TaskType.CAUSAL_LM,
    # bias="none"  # Add this for better performance
)

In [12]:
from trl import SFTTrainer, SFTConfig

# Follow their epoch-based approach
training_args = SFTConfig(
    output_dir="./llama-3.2-3b-mental-health-fnc_v0.2",  # New version
    
    # Batch size and accumulation (follow their approach)
    per_device_train_batch_size=1,          # Same as theirs
    per_device_eval_batch_size=1,           # Same as theirs
    gradient_accumulation_steps=4,          # Same as theirs
    
    # Learning rate and scheduler (adopt their settings exactly)
    learning_rate=1e-4,                     # Same as theirs
    warmup_ratio=0.1,                       # Same as theirs
    lr_scheduler_type="cosine",             # Same as theirs
    max_grad_norm=1.0,                      # Same as theirs
    weight_decay=0.1,                       # Same as theirs
    
    # Training duration (adjust for your dataset size)
    num_train_epochs=10,                     # More epochs since your dataset is smaller
    
    # Evaluation and logging (follow their approach)
    eval_strategy="epoch",                  # Same as theirs - epoch-based
    logging_steps=5,                        # Same as theirs
    save_strategy="no",                     # Same as theirs - no intermediate saves
    
    # Memory and efficiency (adopt their settings)
    bf16=True,
    gradient_checkpointing=True,            # Save memory
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Sequence handling (adopt their approach)
    max_seq_length=1024,                    # Slightly shorter than their 1500 for stability
    packing=True,                           # Same as theirs
    dataset_text_field="text",
    
    # Reporting
    # report_to="tensorboard",                # Same as theirs
    push_to_hub=False,                      # Same as theirs
)

print("✅ Epoch-based training configuration set (following their approach)")
print(f"📊 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📊 Will evaluate after each epoch")

✅ Epoch-based training configuration set (following their approach)
📊 Effective batch size: 4
📊 Will evaluate after each epoch


In [13]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:412: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your att

In [14]:
# print("🚀 Starting training...")
# trainer.train()

In [15]:
# # Save the final model
# print("💾 Saving model...")
# trainer.save_model()
# tokenizer.save_pretrained(training_args.output_dir)

# print("✅ Training complete!")
# print(f"Model saved to: {training_args.output_dir}")

In [16]:
# # Load the model following their approach
# from peft import PeftModel, PeftConfig
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# # Model paths
# peft_model_id = "./llama-3.2-3b-mental-health-fnc_v0.2"  # Your trained adapter

# print("🔄 Loading model following their approach...")

# # Load PEFT config to get base model info
# config = PeftConfig.from_pretrained(peft_model_id)
# print(f"Base model: {config.base_model_name_or_path}")

# # Load base model
# model = AutoModelForCausalLM.from_pretrained(
#     config.base_model_name_or_path,
#     device_map="auto",
#     torch_dtype=torch.bfloat16
# )

# # Load tokenizer from the adapter (contains special tokens)
# tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
# print(f"Tokenizer vocab size: {len(tokenizer)}")

# # Resize model to accommodate new tokens
# model.resize_token_embeddings(len(tokenizer))

# # Load the adapter weights
# model = PeftModel.from_pretrained(model, peft_model_id)
# model.to(torch.bfloat16)
# model.eval()

# print("✅ Model loaded successfully with their approach!")

In [17]:
# test_sample = test_dataset[0]
# test_sample["text"][]

In [18]:
# # Test the model with the CORRECT tokenizer (the one with special tokens)
# def test_model_v2(patient_id, query, max_length=1024):
#     # Format exactly like training data - include the explicit instruction
#     messages = [
#         {
#             "role": "system",
#             "content": "You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function."
#         },
#         {
#             "role": "user",
#             "content": f"Patient ID: {patient_id}\nQuery: {query}"
#         }
#     ]
    
#     # Apply chat template using the LOADED tokenizer (not a fresh one)
#     prompt = tokenizer.apply_chat_template(
#         messages, 
#         tokenize=False, 
#         add_generation_prompt=True
#     )
    
#     print(f"🧪 Testing query: {query}")
#     print(f"Tokenizer vocab size: {len(tokenizer)}")  # Should show the expanded vocab
    
#     # Tokenize and generate
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_length,
#             do_sample=True,
#             temperature=0.01,           # Lower temperature like theirs
#             top_p=0.95,                 # Like theirs
#             repetition_penalty=1.0,     # Like theirs
#             # pad_token_id=tokenizer.eos_token_id,
#             # eos_token_id=tokenizer.eos_token_id,
            
#         )
    
#     # Decode only new tokens
#     response = tokenizer.decode(
#         outputs[0], 
#         skip_special_tokens=False  # Don't skip special tokens to see <think> and <function>
#     )
    
#     return response

# print("🧪 Testing with the CORRECT tokenizer...")
# print("="*80)

# # Test with the loaded model and tokenizer
# test_query = "I want to balance calorie burn with my mental health for 5 days"
# response = test_model_v2("UID_123", test_query)
# print(f"Response: {response}")
# print("="*80)

In [19]:
# print(tokenizer.get_chat_template())